# Check the clisops regrid operator against CMIP6 QA/QC findings

Version 2026-11-16

These tests use synthetic NetCDF files to check whether clisops produces the same bilinear regridding result when coordinate metadata is changed. Xarray decodes each fixture before the public clisops operator remaps it to a common target grid using xESMF and ESMPy.

| Finding | Metadata issue |
| --- | --- |
| [3.3] | Missing coordinate `axis` or `standard_name` |
| [3.7] | Projected x/y axes with degree units |

The candidates below list the model, experiment, and variant for future real-data validation. This notebook tests only the synthetic fixtures.

In [1]:
import re
from contextlib import redirect_stdout
from importlib.metadata import version
from io import StringIO

import esmpy
import ipytest
import numpy as np
import pytest
import xarray as xr
import xesmf
from IPython.display import Markdown, display
from clisops import CONFIG
from clisops.ops.regrid import regrid

ipytest.autoconfig(raise_on_error=True, magics=False, addopts=())

for package in ("clisops", "xarray", "xesmf", "esmpy"):
    print(f"{package} {version(package)}")

clisops 0.18.1
xarray 2025.10.1
xesmf 0.9.2
esmpy 8.9.1


## Finding [3.3] — Coordinate axis / standard_name metadata

As in the subsetting notebook, each variant removes either `axis` or `standard_name` from one of `time`, `lat`, `lon`, and `plev`. The regular-grid fixtures retain the coordinate names, applicable units, and values. Bilinear regridding should preserve both pressure levels and match the fully attributed control at each target point.

Candidate for future real-data validation — `CNRM-CM6-1` / `historical` / `r1i1p1f2`.

## Finding [3.7] — Horizontal x/y coordinate units

The synthetic variant relabels metre-valued `projection_x_coordinate` and `projection_y_coordinate` axes as `degrees`. Both curvilinear fixtures share valid 2-D latitude and longitude, native axis values, and data. A regular latitude/longitude grid provides a separate control for the operator and backend.

The CMIP6 grid table uses the same `standard_name` for the metre and angular entries on each axis (`x`/`x_deg` and `y`/`y_deg`). CMOR also writes the angular entries as `x` and `y`. To tell whether `degrees` is appropriate in a real file, check the declared CF version, grid mapping, and coordinate values against the intended grid. [CMIP6 grid table](https://github.com/PCMDI/cmip6-cmor-tables/blob/main/Tables/CMIP6_grids.json)

CF 1.8 specified the older names for geostationary scan angles in radians. CF 1.9 specifies `projection_x_angular_coordinate` and `projection_y_angular_coordinate` for those axes and deprecates the older names in that context. The CMIP7 grid table uses the angular names for its degree entries. [CF 1.8, Appendix F](https://cfconventions.org/Data/cf-conventions/cf-conventions-1.8/cf-conventions.html#_geostationary_projection), [CF 1.9, Appendix F](https://cfconventions.org/Data/cf-conventions/cf-conventions-1.9/cf-conventions.html#_geostationary_projection), [CMIP7 grid table](https://github.com/WCRP-CMIP/cmip7-cmor-tables/blob/main/tables/CMIP7_grids.json)

Candidate for future real-data validation — `MIROC6` / `historical` / `r1i1p1f1`.

In [2]:
LAT = {"standard_name": "latitude", "units": "degrees_north"}
LON = {"standard_name": "longitude", "units": "degrees_east"}
time = np.array(["2000-01-01", "2000-01-02"], dtype="datetime64[ns]")
lat = np.array([44.0, 45.0, 46.0, 47.0])
lon = np.array([-12.0, -11.0, -10.0, -9.0])
lon_grid, lat_grid = np.meshgrid(lon, lat)
lat_curved = lat_grid + 0.08 * (lon_grid - lon[0])
lon_curved = lon_grid + 0.06 * (lat_grid - lat[0])


def temperatures(latitude, longitude):
    return 260.0 + latitude + 2.0 * longitude + np.array([0.0, 10.0])[:, None, None]


regular = xr.Dataset(
    {"tas": (("time", "lat", "lon"), temperatures(lat_grid, lon_grid))},
    coords={"time": time, "lat": ("lat", lat, LAT), "lon": ("lon", lon, LON)},
)
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), temperatures(lat_curved, lon_curved))},
    coords={
        "time": time,
        "x": ("x", np.arange(4) * 80_000,
              {"standard_name": "projection_x_coordinate", "units": "m"}),
        "y": ("y", np.arange(4) * 110_000,
              {"standard_name": "projection_y_coordinate", "units": "m"}),
        "lat": (("y", "x"), lat_curved, LAT),
        "lon": (("y", "x"), lon_curved, LON),
    },
)
incorrect_units = curvilinear.copy(deep=True)
for axis in ("x", "y"):
    incorrect_units[axis].attrs["units"] = "degrees"

target = xr.Dataset(coords={
    "lat": ("lat", [44.6, 46.2], LAT),
    "lon": ("lon", [-11.4, -10.4, -9.4], LON),
})

pressure = np.array([100_000., 85_000.])
profile = xr.Dataset(
    {"ta": (("time", "plev", "lat", "lon"),
            regular.tas.values[:, None, :, :] + np.array([0.0, -6.0])[None, :, None, None])},
    coords={
        "time": ("time", time, {"standard_name": "time", "axis": "T"}),
        "plev": ("plev", pressure,
                 {"standard_name": "air_pressure", "axis": "Z", "units": "Pa", "positive": "down"}),
        "lat": ("lat", lat, {**LAT, "axis": "Y"}),
        "lon": ("lon", lon, {**LON, "axis": "X"}),
    },
)
metadata_cases = [
    pytest.param(coordinate, attribute, id=f"3.3-{coordinate}-without-{attribute}")
    for coordinate in ("time", "lat", "lon", "plev")
    for attribute in ("axis", "standard_name")
]

## Pytest tests

Each run gets an empty weights cache so clisops calculates weights independently for each input. Each test checks the target grid, the two input times, and finite values. The 3.3 tests also check that both pressure levels are retained. Each metadata variant is compared with its own fully attributed control using an absolute tolerance of `1e-10`.

The backend imports and the regular-grid test help diagnose environment or operator failures. If a control fails, its metadata comparison is inconclusive.

In [3]:
ipytest.clean()


def remap(source, directory, *, missing=None):
    path = directory / "source.nc"
    source.to_netcdf(path, engine="netcdf4")
    with xr.open_dataset(path, engine="netcdf4") as dataset:
        decoded = dataset.load()

    for axis in ("x", "y"):
        if axis in source.coords:
            assert decoded[axis].attrs["units"] == source[axis].attrs["units"]

    if missing is not None:
        coordinate, attribute = missing
        assert attribute not in decoded[coordinate].attrs

    weights = directory / "weights"
    weights.mkdir()
    with pytest.MonkeyPatch.context() as patch:
        patch.setitem(CONFIG["clisops:grid_weights"], "local_weights_dir", str(weights))
        outputs = regrid(decoded, grid=target.copy(deep=True), method="bilinear", output_type="xarray")
    assert len(outputs) == 1
    return outputs[0]


def assert_target(result, variable="tas", *, levels=None):
    field = result[variable]
    expected_dims = ("time", "lat", "lon")
    expected_sizes = {"time": len(time), "lat": target.lat.size, "lon": target.lon.size}
    if levels is not None:
        expected_dims = ("time", "plev", "lat", "lon")
        expected_sizes["plev"] = len(levels)
        np.testing.assert_array_equal(field.plev, levels)
    assert field.dims == expected_dims
    assert result.sizes == expected_sizes
    np.testing.assert_array_equal(field.time, time)
    np.testing.assert_array_equal(field.lat, target.lat)
    np.testing.assert_array_equal(field.lon, target.lon)
    assert np.isfinite(field.values).all()


@pytest.fixture(scope="module", params=[pytest.param(curvilinear, id="3.7-curvilinear-control")])
def curvilinear_control(request, tmp_path_factory):
    return remap(request.param, tmp_path_factory.mktemp("curvilinear-control"))


@pytest.mark.parametrize("source", [pytest.param(regular, id="3.7-regular-control")])
def test_regular_grid(source, tmp_path):
    assert_target(remap(source, tmp_path))


def test_curvilinear_grid(curvilinear_control):
    assert_target(curvilinear_control)


@pytest.mark.parametrize("source", [pytest.param(incorrect_units, id="3.7-degree-units")])
def test_projected_axis_units(curvilinear_control, source, tmp_path):
    actual = remap(source, tmp_path)
    assert_target(actual)
    np.testing.assert_allclose(actual.tas, curvilinear_control.tas, rtol=0, atol=1e-10)
    difference = np.abs(actual.tas.values - curvilinear_control.tas.values).max()
    print(f"Maximum absolute difference from the curvilinear control: {difference:.3g}")


@pytest.fixture(scope="module")
def profile_control(tmp_path_factory):
    result = remap(profile, tmp_path_factory.mktemp("profile-control"))
    assert_target(result, "ta", levels=pressure)
    return result.ta


@pytest.mark.parametrize("coordinate, attribute", metadata_cases)
def test_missing_coordinate_metadata(profile_control, coordinate, attribute, tmp_path):
    source = profile.copy(deep=True)
    del source[coordinate].attrs[attribute]
    result = remap(source, tmp_path, missing=(coordinate, attribute))
    assert_target(result, "ta", levels=pressure)
    np.testing.assert_allclose(result.ta, profile_control, rtol=0, atol=1e-10)

## Run pytest

In [4]:
FINDINGS = {
    "3.3": "Coordinate axis / standard_name metadata",
    "3.7": "Horizontal x/y coordinate units",
}


class FindingResults:
    def __init__(self):
        self.tests = {finding: [] for finding in FINDINGS}
        self.outcomes = {}

    def pytest_itemcollected(self, item):
        # A shared control can contribute to more than one finding.
        for finding in item.callspec.id.split("-", 1)[0].split("+"):
            self.tests[finding].append(item.nodeid)

    def pytest_runtest_logreport(self, report):
        phase_outcomes = self.outcomes.setdefault(report.nodeid, {})
        phase_outcomes[report.when] = "skipped" if hasattr(report, "wasxfail") else report.outcome

    def status(self, finding):
        outcomes = [self.outcomes.get(nodeid, {}) for nodeid in self.tests[finding]]
        passed = {"setup": "passed", "call": "passed", "teardown": "passed"}
        if any("failed" in phases.values() for phases in outcomes):
            return "FAIL"
        if outcomes and all(phases == passed for phases in outcomes):
            return "PASS"
        return "INCOMPLETE"


results = FindingResults()
with redirect_stdout(StringIO()) as report:
    ipytest.run(
        "-v", "-rP", "--tb=long", "--no-header", "--color=yes", "-p", "no:cacheprovider",
        plugins=[results], raise_on_error=False,
    )

# Replace generated module names and kernel paths with readable notebook locations.
output = re.sub(r"t_[0-9a-f]{32}\.py::", "", report.getvalue())
output = re.sub(r"t_[0-9a-f]{32}\.py(?=:)", "Notebook tests", output)
output = re.sub(
    r"^((?:\x1b\[[0-9;]*m)*).*[/\\]ipykernel_\d+[/\\]\d+\.py(?=(?:\x1b\[[0-9;]*m)*:\d+)",
    r"\1Notebook cell", output, flags=re.MULTILINE,
)
print(output, end="")

======================================= test session starts ========================================
collecting ... collected 11 items

test_regular_grid[3.7-regular-control] PASSED         [  9%]
test_curvilinear_grid[3.7-curvilinear-control] PASSED [ 18%]
test_projected_axis_units[3.7-curvilinear-control-3.7-degree-units] PASSED [ 27%]
test_missing_coordinate_metadata[3.3-time-without-axis] PASSED [ 36%]
test_missing_coordinate_metadata[3.3-time-without-standard_name] PASSED [ 45%]
test_missing_coordinate_metadata[3.3-lat-without-axis] PASSED [ 54%]
test_missing_coordinate_metadata[3.3-lat-without-standard_name] PASSED [ 63%]
test_missing_coordinate_metadata[3.3-lon-without-axis] PASSED [ 72%]
test_missing_coordinate_metadata[3.3-lon-without-standard_name] PASSED [ 81%]
test_missing_coordinate_metadata[3.3-plev-without-axis] PASSED [ 90%]
test_missing_coordinate_metadata[3.3-plev-without-standard_name] PASSED [100%]

========================================= warnings summary ========

## Results and limitations

The 3.3 cases use a rectilinear grid with conventional coordinate names and latitude/longitude units. Files with unfamiliar coordinate names or several missing attributes need separate validation.

The two curvilinear fixtures produced identical values (maximum absolute difference of `0`). For these inputs xESMF builds the bilinear weights from the 2-D auxiliary latitude and longitude. Changing the `units` attribute on the 1-D `x` and `y` axes leaves that geometry unchanged. The fixture uses increasing axes and tests only the effect of the units attribute.

The table shows the previously reported CMIP6 QA/QC severity. For 3.7, the reviewer indicated that the newer coordinate checks may classify confirmed unit errors as `HIGH` and that those checks are not yet run for CMIP6. The regridding result measures interpolation behavior. Assessing metadata compliance requires inspecting the source file.

Bilinear regridding succeeded despite warnings about missing latitude and longitude bounds. Other methods may require valid bounds. Real CMIP6 grids still need validation.

In [5]:
table = [
    "| Finding | QA/QC severity | Regrid | Woodpecker |",
    "| --- | --- | --- | --- |",
]
for finding, description in FINDINGS.items():
    status = results.status(finding)
    woodpecker = "Not needed (synthetic)" if status == "PASS" else "To investigate"
    table.append(f"| [{finding}] {description} | MEDIUM | {status} | {woodpecker} |")

display(Markdown("\n".join(table)))

| Finding | QA/QC severity | Regrid | Woodpecker |
| --- | --- | --- | --- |
| [3.3] Coordinate axis / standard_name metadata | MEDIUM | PASS | Not needed (synthetic) |
| [3.7] Horizontal x/y coordinate units | MEDIUM | PASS | Not needed (synthetic) |